In [10]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.impute import KNNImputer

In [20]:
df = pd.read_csv("data/data.csv")

# How to handle the rows with missing columns
#  - Fill with zero
#  - Do a k nearest neighbors
#     - Sklearn learns other fighters that are similar to that fighter and fills the missing column with an average (or something like that)

#one hot encoding for categorical features (stance, weight class), title_bout column into 1s and 0s (boolean)
#dont use fighter name in training features, only as a way to distinguish rows
#sort df based on date, make sure training fights occured before test fights
#want to look at target labels one at a time

In [21]:
# Load dataset
df = pd.read_csv("data/data.csv")
df.head()

,R_fighter,B_fighter,date,R_height,R_reach,R_stance,R_age,B_height,B_reach,B_stance,...,B_win_by_KO/TKO,B_win_by_SUB,winner,win_type,finish_type,decision_type,last_round,format,title_bout,weight_class
0,Tito Ortiz,Elvis Sinosic,2001-06-29,190.50,74.0,Orthodox,26,190.50,77.0,Orthodox,...,0.0,1.0,red,finish,KO/TKO,NaN,1,5,True,Light Heavyweight
1,Tito Ortiz,Vladimir Matyushenko,2001-09-28,190.50,74.0,Orthodox,26,182.88,74.0,Orthodox,...,0.0,0.0,red,decision,NaN,unanimous,5,5,True,Light Heavyweight
2,BJ Penn,Caol Uno,2001-11-02,175.26,70.0,Orthodox,22,170.18,70.0,Southpaw,...,0.5,0.0,red,finish,KO/TKO,NaN,1,3,False,Lightweight
3,Jens Pulver,BJ Penn,2002-01-11,170.18,70.0,Southpaw,27,175.26,70.0,Orthodox,...,1.0,0.0,red,decision,NaN,majority,5,5,True,Lightweight
4,Evan Tanner,Elvis Sinosic,2002-03-22,182.88,74.0,Orthodox,31,190.50,77.0,Orthodox,...,0.0,0.5,red,finish,KO/TKO,NaN,1,3,False,Light Heavyweight


In [22]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort fights so earlier fights come first
df = df.sort_values("date").reset_index(drop=True)

# Convert title_bout to numeric (True/False → 1/0)
df["title_bout"] = df["title_bout"].astype(int)

In [23]:
# Separate Target and Features
# Target label
y = df["winner"]

# Drop columns that won't be used as features
X = df.drop(columns=[
    "winner",
    "win_type",
    "finish_type",
    "decision_type",
    "R_fighter",
    "B_fighter",
    "date"
])

In [24]:
# Categorical columns
cat_cols = ["R_stance", "B_stance", "weight_class"]

# Numeric columns
num_cols = X.columns.drop(cat_cols)

# Preprocessing pipeline
transformer = make_column_transformer(
    (KNNImputer(n_neighbors=5), num_cols),               # fill missing numeric values
    (OneHotEncoder(handle_unknown="ignore"), cat_cols),  # encode categorical
    remainder="drop"
)

# Fit & transform the features
X_transformed = transformer.fit_transform(X)

X_transformed.shape

(5885, 142)

In [ ]:
#RYANS CODE

# Code for data exploring:

dataframe = pd.read_csv("data/data.csv")

# Train/Test split (manually sorted chronologically by date)
dataframe = dataframe.sort_values("date")
split_index = int(len(dataframe) * 0.9)

train_df = dataframe.iloc[:split_index]
test_df = dataframe.iloc[split_index:]
#print("TRAIN DF stats:\n", train_df.shape, train_df.dtypes)
#print("\n\nTEST DF stats:\n", test_df.shape, test_df.dtypes)

# We have multiple possible labels depending on which stage we're training.
target_cols = ["winner", "win_type", "finish_type", "decision_type", "last_round"]

# Excluding these features from model
id_columns =  ["R_fighter", "B_fighter", "date"]

# We want to include all other columns:
feature_cols = [c for c in dataframe.columns if c not in target_cols + id_columns]

train_features = train_df[feature_cols]
test_features = test_df[feature_cols]
# Just stage 1 for now (winner prediction: red, blue, or draw)
train_labels_winner = train_df["winner"]
test_labels_winner = test_df["winner"]

# Check for missing values (Look)
missing_by_col = train_features.isna().sum()
cols_with_missing = missing_by_col[missing_by_col > 0].sort_values(ascending=False)
print(f"\nTotal missing values in train_features: {train_features.isna().sum().sum()}")
print(f"\nColumns with missing values ({len(cols_with_missing)} columns):")
print(cols_with_missing.head(20))